In [4]:
import pandas as pd
import numpy as np
import pickle

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

In [5]:
movies = pd.read_csv(
    "movies_metadata.csv",
    low_memory=False
)

ratings = pd.read_csv(
    "ratings.csv"
)

links = pd.read_csv(
    "links.csv"
)

In [6]:
movies['id'] = pd.to_numeric(
    movies['id'],
    errors='coerce'
)

movies = movies.dropna(
    subset=['id']
)

movies['id'] = movies['id'].astype(int)

In [7]:
links['tmdbId'] = pd.to_numeric(
    links['tmdbId'],
    errors='coerce'
)

links = links.dropna(
    subset=['tmdbId']
)

links['tmdbId'] = links[
    'tmdbId'
].astype(int)

In [8]:
ratings = ratings.merge(
    links,
    on='movieId'
)

In [9]:
ratings = ratings.astype({

    'userId': 'int32',

    'movieId': 'int32',

    'tmdbId': 'int32',

    'rating': 'float32'
})

In [10]:
# FILTER ACTIVE USERS

user_rating_counts = ratings.groupby(
    'userId'
)['rating'].count()

active_users = user_rating_counts[
    user_rating_counts >= 200
].index

ratings = ratings[
    ratings['userId'].isin(
        active_users
    )
]

In [11]:
# FILTER POPULAR MOVIES

movie_rating_counts = ratings.groupby(
    'tmdbId'
)['rating'].count()

popular_movies = movie_rating_counts[
    movie_rating_counts >= 200
].index

ratings = ratings[
    ratings['tmdbId'].isin(
        popular_movies
    )
]

In [12]:
print(ratings.shape)

(14997033, 6)


In [13]:
movies = movies[
    [
        'id',
        'title',
        'overview',
        'genres',
        'poster_path',
        'vote_average',
        'release_date',
        'runtime'
    ]
]

In [14]:
movie_user_matrix = ratings.pivot_table(
    index='tmdbId',
    columns='userId',
    values='rating'
)

In [15]:

movie_user_matrix.fillna(
    0,
    inplace=True
)

In [16]:
sparse_matrix = csr_matrix(
    movie_user_matrix.values
)

In [17]:
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=11,
    n_jobs=-1
)

model.fit(sparse_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=11)

In [18]:
movie_id_to_index = {

    movie_id: index

    for index, movie_id

    in enumerate(
        movie_user_matrix.index
    )
}

In [19]:
def recommend_collaborative(movie_name):

    matches = movies[
        movies['title']
        .fillna('')
        .str.lower()
        .str.contains(
            movie_name.lower()
        )
    ]

    if matches.empty:

        print("Movie not found")

        return []

    movie_id = matches.iloc[0]['id']

    print("TMDB Movie ID:", movie_id)

    if movie_id not in movie_id_to_index:

        print("Movie not present in collaborative matrix")

        return []

    movie_index = movie_id_to_index[
        movie_id
    ]

    distances, indices = model.kneighbors(
        sparse_matrix[movie_index],
        n_neighbors=11
    )

    recommendations = []

    for i in range(
        1,
        len(indices[0])
    ):

        similar_movie_index = indices[0][i]

        similar_movie_id = (
            movie_user_matrix.index[
                similar_movie_index
            ]
        )

        movie_data = movies[
            movies['id']
            == similar_movie_id
        ]

        if movie_data.empty:
            continue

        movie_data = movie_data.iloc[0]

        poster_url = None

        if (
            pd.notna(
                movie_data['poster_path']
            )
            and str(
                movie_data['poster_path']
            ).startswith("/")
        ):

            poster_url = (
                "https://image.tmdb.org/t/p/w500"
                + movie_data[
                    'poster_path'
                ]
            )

        recommendations.append({

            "id":
                int(movie_data['id']),

            "title":
                movie_data['title'],

            "overview":
                movie_data['overview'],

            "poster":
                poster_url,

            "rating":
                (
                    round(
                        float(
                            movie_data[
                                'vote_average'
                            ]
                        ),
                        1
                    )
                    if pd.notna(
                        movie_data[
                            'vote_average'
                        ]
                    )
                    else None
                ),

            "release_date":
                movie_data[
                    'release_date'
                ],

            "runtime":
                (
                    int(
                        movie_data[
                            'runtime'
                        ]
                    )
                    if pd.notna(
                        movie_data[
                            'runtime'
                        ]
                    )
                    else None
                ),

            "distance":
                round(
                    float(
                        distances[0][i]
                    ),
                    3
                )
        })

    return recommendations

In [20]:
recommend_collaborative(
    "Inception"
)

TMDB Movie ID: 27205


[{'id': 155,
  'title': 'The Dark Knight',
  'overview': 'Batman raises the stakes in his war on crime. With the help of Lt. Jim Gordon and District Attorney Harvey Dent, Batman sets out to dismantle the remaining criminal organizations that plague the streets. The partnership proves to be effective, but they soon find themselves prey to a reign of chaos unleashed by a rising criminal mastermind known to the terrified citizens of Gotham as the Joker.',
  'poster': 'https://image.tmdb.org/t/p/w500/1hRoyzDtpgMU7Dz4JF22RANzQO7.jpg',
  'rating': 8.3,
  'release_date': '2008-07-16',
  'runtime': 152,
  'distance': 0.202},
 {'id': 19995,
  'title': 'Avatar',
  'overview': 'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.',
  'poster': 'https://image.tmdb.org/t/p/w500/kmcqlZGaSh20zpTbuoF0Cdn07dT.jpg',
  'rating': 7.2,
  'release_date': '2009-12-10',
  'runtime': 162,


In [21]:
pickle.dump(
    movies,
    open(
        'collaborative_movies.pkl',
        'wb'
    )
)

pickle.dump(
    model,
    open(
        'collaborative_model.pkl',
        'wb'
    )
)

pickle.dump(
    sparse_matrix,
    open(
        'collaborative_sparse.pkl',
        'wb'
    )
)

# pickle.dump(
#     movie_user_matrix,
#     open(
#         'movie_user_matrix.pkl',
#         'wb'
#     )
# )

pickle.dump(
    movie_id_to_index,
    open(
        'movie_id_to_index.pkl',
        'wb'
    )
)

movie_ids = list(
    movie_user_matrix.index
)

pickle.dump(
    movie_ids,
    open(
        "movie_ids.pkl",
        "wb"
    )
)